In [2]:
import pandas as pd
from pathlib import Path

In [5]:
interim_df = pd.read_parquet("../data/interim/lastfm/lastfm_scrobbles_merged.parquet")

In [6]:
interim_df

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007
...,...,...,...,...
559444,Perfume Genius,No Front Teeth,NaN,2026
559445,Perfume Genius,No Front Teeth,NaN,2026
559446,Perfume Genius,No Front Teeth,NaN,2026
559447,Perfume Genius,No Front Teeth,NaN,2026


In [10]:
# 1. Drop rows with missing timestamps
df = interim_df.dropna(subset=["timestamp"])
df

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007
...,...,...,...,...
558996,Alaskan Tapes,Wait,1767473119,2026
558997,Brian Eno,An Ending (Ascent) - Remastered 2005,1767472882,2026
558998,Michael Andrews,Boy Moves The Sun,1767472698,2026
558999,Stellardrone,Tranquility,1767472374,2026


In [9]:
# 2. Convert timestamps to datetime and extract year and month
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["timestamp"] = df["timestamp"].astype(int)
df["played_at"] = pd.to_datetime(df["timestamp"], unit="s")
df["year"] = df["played_at"].dt.year
df["month"] = df["played_at"].dt.to_period("M")
df

,artist,track,timestamp,year_file,played_at,year,month
0,Oasis,Don't Look Back in Anger,1199052426,2007,2007-12-30 22:07:06,2007,2007-12
1,Oasis,Wonderwall,1199052167,2007,2007-12-30 22:02:47,2007,2007-12
2,Oasis,Don't Look Back in Anger,1199051867,2007,2007-12-30 21:57:47,2007,2007-12
3,Oasis,Wonderwall,1199051609,2007,2007-12-30 21:53:29,2007,2007-12
4,Incubus,Aqueous Transmission,1199051161,2007,2007-12-30 21:46:01,2007,2007-12
...,...,...,...,...,...,...,...
558996,Alaskan Tapes,Wait,1767473119,2026,2026-01-03 20:45:19,2026,2026-01
558997,Brian Eno,An Ending (Ascent) - Remastered 2005,1767472882,2026,2026-01-03 20:41:22,2026,2026-01
558998,Michael Andrews,Boy Moves The Sun,1767472698,2026,2026-01-03 20:38:18,2026,2026-01
558999,Stellardrone,Tranquility,1767472374,2026,2026-01-03 20:32:54,2026,2026-01


In [15]:
# 4. Sanity check
print(f"{df.shape} records after processing.")
print(f"Number of empty rows {df.isna().sum()}")
print(f"Number of unique artists in the df {df.artist.nunique()}")
print(f"Number of unique tracks in the df {df.track.nunique()}")
print(f"Top rows {df.head()}")

(550151, 4) records after processing.
Number of empty rows artist       0
track        0
timestamp    0
year_file    0
dtype: int64
Number of unique artists in the df 13837
Number of unique tracks in the df 69106
Top rows     artist                     track   timestamp  year_file
0    Oasis  Don't Look Back in Anger  1199052426       2007
1    Oasis                Wonderwall  1199052167       2007
2    Oasis  Don't Look Back in Anger  1199051867       2007
3    Oasis                Wonderwall  1199051609       2007
4  Incubus      Aqueous Transmission  1199051161       2007


In [18]:
# 5. Deduplication
df.duplicated(subset=["artist", "track", "timestamp"]).sum()
df = df.drop_duplicates(subset=["artist", "track", "timestamp"])
df

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007
...,...,...,...,...
558996,Alaskan Tapes,Wait,1767473119,2026
558997,Brian Eno,An Ending (Ascent) - Remastered 2005,1767472882,2026
558998,Michael Andrews,Boy Moves The Sun,1767472698,2026
558999,Stellardrone,Tranquility,1767472374,2026


In [62]:
# 6. Cleaning artist names
import re

def clean_text_artist(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # remove brackets (artist rarely has any important info in brackets)
    text = re.sub(r"\(.*?\)", "", text)
    
    # remove strange suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # remove special characters
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

In [63]:
# 7. Cleaning track names

def clean_text_track(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # Remove content in parentheses that is not part of the title
    text = re.sub(
        r"\((?:"
        r"feat\.?.*?|ft\.?.*?|"
        r"remaster(?:ed)?(?: \d{4})?|"
        r"live(?: at .*?)?|"
        r"radio edit|edit|"
        r"(?:acoustic|alternative|instrumental)(?: version)?|"
        r"mix|version|"
        r"deluxe|bonus track|"
        r"single version"
        r")\)",
        "",
        text,
    )
    
    # remove feat/ft also if outside brackets
    text = re.sub(r"\b(feat|ft)\.?\b.*", "", text)
    
    # 🔥 remove remix/rework/edit/version also if outside brackets
    text = re.sub(r"\b(remix|rework|edit|version)\b.*", "", text)
    
    # remove strange suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # remove special characters
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None

In [64]:
df["artist_clean"] = df["artist"].apply(clean_text_artist)
df["track_clean"] = df["track"].apply(clean_text_track)

In [68]:
df = df.dropna(subset=["track_clean"])

In [71]:
print("Before:", df["track"].nunique())
print("After:", df["track_clean"].nunique())

print("Before:", df["artist"].nunique())
print("After:", df["artist_clean"].nunique())

Before: 68803
After: 58535
Before: 13773
After: 13701


In [73]:
df[["track", "track_clean"]].sample(50)

,track,track_clean
424822,Brother the Cloud,brother the cloud
462697,Only A Moment,only a moment
296629,To Be Loved,to be loved
165569,Diamonds,diamonds
281307,Wake Up Call,wake up call
327671,Anywhere,anywhere
321428,Planet Caravan,planet caravan
368662,Wild Horses - Live / Remastered 2009,wild horses
41116,Timshel,timshel
139781,Pet,pet


In [61]:
df[["artist", "artist_clean"]].sample(50)

,artist,artist_clean
459776,Japanese Breakfast,japanese breakfast
348134,Minor Victories,minor victories
491370,Japanese Breakfast,japanese breakfast
540285,Oberhofer,oberhofer
538874,Autechre,autechre
489649,Japanese Breakfast,japanese breakfast
246659,The Smiths,the smiths
485718,Deftones,deftones
183780,BANKS,banks
54756,Joy Division,joy division


In [49]:
df["parentheses"] = df["track"].apply(lambda x: re.findall(r"\(.*?\)", x.lower()))

In [52]:
df["parentheses"].explode().value_counts().head(30)

parentheses
(live)                                     648
(remastered)                               618
(acoustic)                                 551
(bonus track)                              400
(a deal with god)                          261
(don't fear)                               233
(feat. jack liebeck)                       216
(demo)                                     208
(cosi fan tutte)                           190
(album version)                            177
(instrumental)                             163
(in the house of flies)                    157
(omnimotion feat. krister linder remix)    151
(far away)                                 144
(intro)                                    142
(single version)                           130
(feat. bruce springsteen)                  130
(feat. alison mosshart)                    110
(ascent)                                   109
(reprise)                                  106
(radio edit)                                95
(

In [74]:
df.to_parquet(
    "../data/processed/lastfm_scrobbles_clean.parquet",
    index=False
)